# Qwen3.5-9B-Base LoRA — two-stage (`pool_v1` → host)

Same two-stage LoRA24 recipe as the 4B/8B pool runs (L **0.544**, E **0.564**), but the trunk is **Qwen3.5-9B-Base**: hybrid Gated DeltaNet + gated attention, **32** layers, hidden **4096**, `model_type=qwen3_5_text`.

1. **Stage 1:** 1 epoch on **`pool_v1.parquet`** (≥3 models / prompt), class-weighted CE.
2. **Stage 2:** 2 epochs on host `train.csv` at lower LR; folds `0–4` + `final`.
3. **Submit:** plain + centred twin + Sinkhorn → `submission.parquet`.

Kaggle’s default `transformers` does **not** know Qwen3.5. Run the setting notebook first (`transformers>=5.9`). A T4 cannot hold 9B fp16 plus activations, so this loads the first **24** layers in **4-bit NF4**.

Prefer a **text** / **Base** checkpoint (`qwen3_5_text`). Skip Instruct and any vision wrapper (`preprocessor_config.json`).

## Runtime (2× T4)
| Phase | Work | Estimate |
| --- | --- | --- |
| Stage 1 | 1 epoch on ~35k, 1 GPU, 4-bit 9B/24 | ~4–6 h |
| Stage 2 | 2 epochs host, 2 GPUs | ~1.5–2.5 h |
| Submit | `job_final` on `test.csv` | minutes |
| **Total** | | **~6–9 h** |

## Attach on Kaggle (internet off, GPU on)
- Competition **Name That LLM**
- **Qwen3.5-9B** at `/kaggle/input/models/danbth/qwen3-5-9b/transformers/default/1` (set as `MODEL_DIR` in the Hyperparameters cell)
- Dataset **`nishantdahal/ntl-two-stage-lora`** (`pool_v1.parquet`)
- Output of [`Qwen3.5-9B-setting.ipynb`](Qwen3.5-9B-setting.ipynb)
- **2× T4**


## 0. Install deps from the setting notebook

Qwen3.5 needs `transformers>=5.9`. Installs wheels from the setting notebook with `--no-index`.


In [ ]:
import glob
import os
import subprocess
import sys
from pathlib import Path

PKGS = [
    "transformers==5.9.0",
    "bitsandbytes==0.46.1",
    "accelerate==1.6.0",
    "peft==0.15.2",
]

SETTING_INPUTS = [
    Path("/kaggle/input/notebooks/zuhairsan/qwen3-5-9b-setting"),
]


def _has_wheels(p: Path) -> bool:
    return p.is_dir() and any(p.glob("*.whl"))


def find_wheel_dir():
    hits = []
    for root in SETTING_INPUTS:
        if not root.exists():
            continue
        if _has_wheels(root):
            hits.append(root)
        for p in root.rglob("wheels"):
            if _has_wheels(p):
                hits.append(p)
        if _has_wheels(root / "wheels"):
            hits.append(root / "wheels")
    for root in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not root.exists():
            continue
        for p in root.rglob("wheels"):
            if _has_wheels(p):
                hits.append(p)
    # prefer dirs that actually contain a 0.46+ bitsandbytes wheel
    for p in hits:
        names = [x.name for x in p.glob("*.whl")]
        if any(n.startswith("bitsandbytes-0.46") or n.startswith("bitsandbytes-0.47") or n.startswith("bitsandbytes-0.48") or n.startswith("bitsandbytes-0.49") for n in names):
            return p
    if hits:
        return hits[0]
    raise FileNotFoundError(
        "No wheels/ folder. Re-run Qwen3.5-9B-setting.ipynb with internet ON "
        "(needs bitsandbytes>=0.46.1), then Add Input /kaggle/input/notebooks/zuhairsan/qwen3-5-9b-setting."
    )


wheel_dir = find_wheel_dir()
print("wheels", wheel_dir, "n=", len(list(wheel_dir.glob("*.whl"))), flush=True)
print("whl", sorted(p.name for p in wheel_dir.glob("*.whl"))[:30], flush=True)
cmd = [
    sys.executable, "-m", "pip", "install",
    "--no-index", "--find-links", str(wheel_dir),
    "-q", *PKGS,
]
print("+", " ".join(cmd), flush=True)
subprocess.check_call(cmd)
import bitsandbytes, peft, transformers
print("transformers", transformers.__version__, "peft", peft.__version__, "bnb", bitsandbytes.__version__)
assert tuple(int(x) for x in transformers.__version__.split(".")[:2]) >= (5, 9), transformers.__version__
bnb_maj = tuple(int(x) for x in bitsandbytes.__version__.split(".")[:2])
assert bnb_maj >= (0, 46), f"bitsandbytes {bitsandbytes.__version__} < 0.46.1; re-run the setting notebook"


## Hyperparameters


In [ ]:
SEED = 7
FOLD_SEED = 0

# Stage 1: public pool pretrain
S1_EPOCHS = 1
S1_LR_ADAPTER = 1e-4
S1_LR_HEAD = 1e-3
S1_USE_CLASS_WEIGHTS = True

# Stage 2: host fine-tune (lower LR, fewer epochs than host-only 3-epoch run)
S2_EPOCHS = 2
S2_LR_ADAPTER = 5e-5
S2_LR_HEAD = 5e-4
S2_JOBS = {
    0: ['0', '2', '4'],       # GPU 0
    1: ['1', '3', 'final'],  # GPU 1
}

# Shared recipe (match host LoRA24)
LAYERS = [2, 4, 6, 8, 12, 16, 20, 24]
KEEP = 24
MAX_LEN = 447
ACCUM = 6
WD = 0.01
WARMUP = 0.10
R = 16
ALPHA = 32
LDROP = 0.05
HIDDEN = 4096
MODEL_DIR = '/kaggle/input/models/danbth/qwen3-5-9b/transformers/default/1'

import glob
import json
import os
import pathlib
import subprocess
import sys
import time

import numpy as np
import pandas as pd


## Prepare the pool CSV

Loads `pool_v1.parquet`, keeps prompts with answers from at least **3** models (triplets need 3), and writes `pool_train.csv` (`id, prompt_id, text, model`) for the stage-1 worker.


In [ ]:
POOL_PATH = pathlib.Path('/kaggle/input/datasets/nishantdahal/ntl-two-stage-lora/pool_v1.parquet')


def find_pool() -> pathlib.Path:
    if POOL_PATH.is_file():
        return POOL_PATH
    for root in (pathlib.Path('/kaggle/input'), pathlib.Path('data'), pathlib.Path('../data')):
        if root.is_dir():
            hits = sorted(root.rglob('pool_v1.parquet'))
            if hits:
                return hits[0]
    raise FileNotFoundError('pool_v1.parquet not found - attach nishantdahal/ntl-two-stage-lora')


pool_path = find_pool()
pool = pd.read_parquet(pool_path)
need = {'id', 'prompt_id', 'text', 'model'}
missing = need - set(pool.columns)
assert not missing, missing

n_per = pool.groupby('prompt_id')['model'].nunique()
ok = set(n_per[n_per >= 3].index)
stage1_df = pool[pool['prompt_id'].isin(ok)][['id', 'prompt_id', 'text', 'model']].dropna().copy()
stage1_df['text'] = stage1_df['text'].astype(str)
assert stage1_df.model.nunique() == 18, sorted(stage1_df.model.unique())
stage1_df.to_csv('pool_train.csv', index=False)

print('pool source', pool_path)
print('pool rows', len(pool), '-> stage1 rows', len(stage1_df), 'prompts', stage1_df.prompt_id.nunique())
if 'source' in pool.columns:
    print(pool.loc[stage1_df.index, 'source'].value_counts().to_string())
print(stage1_df.model.value_counts().sort_index().to_string())


## Worker script

Same dual-GPU idea as the host-only notebook. Extras:

- `init_adapter`: continue from stage‑1 LoRA (+ optional heads/norms)
- `class_weights`: inverse-frequency weights for unbalanced extended data
- `out_root`: where to write `job_*` / `stage1` folders


### Worker 1 — imports, data, folds, tokenize


In [ ]:
%%writefile worker.py
import importlib.util as _iu
import json
import math
import os
import random
import sys
import time

import numpy as np
import pandas as pd
import torch
from safetensors.torch import load_file, save_file
from torch import nn
from transformers import AutoConfig, AutoModel, AutoTokenizer, BitsAndBytesConfig

_fs = _iu.find_spec
_iu.find_spec = lambda name, *a, **k: None if name.split('.')[0] == 'torchao' else _fs(name, *a, **k)

from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
import peft.import_utils as _pi

_pi.is_torchao_available = lambda *a, **k: False

cfg = json.loads(sys.argv[1])
jobs = sys.argv[2].split(',')

train = pd.read_csv(cfg['train_csv'])
LABELS = sorted(train.model.unique())
label_to_i = {l: i for i, l in enumerate(LABELS)}
y = train.model.map(label_to_i).values
pid = train.prompt_id.values

prompts = np.array(sorted(train.prompt_id.unique()))
rng0 = np.random.RandomState(cfg['fold_seed'])
rng0.shuffle(prompts)
fold = train.prompt_id.map({p: i % 5 for i, p in enumerate(prompts)}).values

tok = AutoTokenizer.from_pretrained(cfg['model_dir'])
tok.padding_side = 'right'
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

tokens = [
    ids + [tok.eos_token_id]
    for ids in tok(
        train.text.tolist(),
        add_special_tokens=False,
        truncation=True,
        max_length=cfg['max_len'],
    )['input_ids']
]

L = cfg['layers']
dev = torch.device('cuda:0')

# class weights (stage 1)
ce_weight = None
if cfg.get('class_weights'):
    counts = train.model.value_counts().reindex(LABELS).fillna(1).values.astype(np.float64)
    w = counts.sum() / (len(LABELS) * counts)
    ce_weight = torch.tensor(w, dtype=torch.float32, device=dev)
    print('class_weights', {l: float(x) for l, x in zip(LABELS, w)}, flush=True)


### Worker 2 — `triplets`


In [ ]:
%%writefile -a worker.py

def triplets(rows, rng):
    out = []
    for p in np.unique(pid[rows]):
        m = rng.permutation(rows[pid[rows] == p])
        if len(m) < 3:
            continue
        for s in range(0, len(m), 3):
            b = list(m[s:s + 3])
            while len(b) < 3:
                b.append(int(rng.choice([x for x in m if x not in b])))
            out.append(b)
    if not out:
        raise RuntimeError('no triplets — need >=3 answers per prompt in the fit set')
    out = [out[i] for i in rng.permutation(len(out))]
    while len(out) % cfg['accum']:
        out.append(out[rng.randint(len(out))])
    return out


### Worker 3 — job setup (optional `init_adapter` + head warm-start)


In [ ]:
%%writefile -a worker.py

out_root = cfg.get('out_root', 'lora_out')

for job in jobs:
    t0 = time.time()
    outdir = os.path.join(out_root, f'job_{job}')
    os.makedirs(outdir, exist_ok=True)

    random.seed(cfg['seed'])
    np.random.seed(cfg['seed'])
    torch.manual_seed(cfg['seed'])
    torch.cuda.manual_seed_all(cfg['seed'])
    rng = np.random.RandomState(cfg['seed'] + (99 if job in ('final', 'stage1') else int(job)))

    if job in ('final', 'stage1'):
        fit = np.arange(len(train))
        valid = np.arange(min(3, len(train)))  # dummy for stage1/final path
    else:
        fit = np.where(fold != int(job))[0]
        valid = np.where(fold == int(job))[0]

    mc = AutoConfig.from_pretrained(cfg['model_dir'])
    mc.num_hidden_layers, mc.use_cache = cfg['keep'], False
    if getattr(mc, 'layer_types', None) is not None:
        mc.layer_types = mc.layer_types[:cfg['keep']]

    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    print(f'job {job} loading 4-bit keep={cfg["keep"]} type={mc.model_type} {cfg["model_dir"]}', flush=True)
    base = AutoModel.from_pretrained(
        cfg['model_dir'],
        config=mc,
        quantization_config=quant,
        device_map={'': 0},
        local_files_only=True,
        low_cpu_mem_usage=True,
        torch_dtype=torch.float16,
    )
    if hasattr(base, 'language_model') and not hasattr(base, 'layers'):
        print(f'job {job}: wrapping language_model (VL checkpoint)', flush=True)
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    try:
        base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    except Exception as e:
        print('grad ckpt skip', type(e).__name__, e, flush=True)
    base.enable_input_require_grads()

    lora_cfg = LoraConfig(
        r=cfg['r'],
        lora_alpha=cfg['alpha'],
        lora_dropout=cfg['ldrop'],
        bias='none',
        target_modules=[
            'q_proj', 'k_proj', 'v_proj', 'o_proj',
            'gate_proj', 'up_proj', 'down_proj',
            'in_proj_qkv', 'in_proj_z', 'in_proj_b', 'in_proj_a', 'out_proj',
        ],
    )
    init_adapter = cfg.get('init_adapter')
    if init_adapter:
        model = PeftModel.from_pretrained(base, init_adapter, is_trainable=True)
        print(f'job {job}: loaded init_adapter {init_adapter}', flush=True)
    else:
        model = get_peft_model(base, lora_cfg)

    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()

    norms = nn.ModuleList([nn.LayerNorm(cfg['hidden']) for _ in L]).to(dev).float()
    plain = nn.Linear(cfg['hidden'] * len(L), len(LABELS)).to(dev).float()
    centered = nn.Linear(cfg['hidden'] * len(L), len(LABELS)).to(dev).float()
    drop = nn.Dropout(0.1)

    # warm-start heads/norms from stage1 when present
    if init_adapter:
        npath = os.path.join(init_adapter, '..', 'norms.safetensors')
        # init_adapter is .../job_stage1/adapter → norms at .../job_stage1/norms.safetensors
        stage_dir = os.path.dirname(init_adapter.rstrip('/\\'))
        nfile = os.path.join(stage_dir, 'norms.safetensors')
        rfile = os.path.join(stage_dir, 'readouts.npz')
        if os.path.isfile(nfile):
            nsd = load_file(nfile)
            norms.load_state_dict({k.split('norms.', 1)[-1]: v for k, v in nsd.items()})
            print(f'job {job}: loaded norms from {nfile}', flush=True)
        if os.path.isfile(rfile):
            with np.load(rfile, allow_pickle=False) as z:
                classes = [str(x) for x in z['classes'].tolist()]
                assert classes == LABELS, (classes, LABELS)
                with torch.no_grad():
                    plain.weight.copy_(torch.tensor(z['plain_coef']))
                    plain.bias.copy_(torch.tensor(z['plain_intercept']))
                    centered.weight.copy_(torch.tensor(z['center_coef']))
                    centered.bias.copy_(torch.tensor(z['center_intercept']))
            print(f'job {job}: loaded heads from {rfile}', flush=True)

    ad = [p for p in model.parameters() if p.requires_grad]
    hd = list(norms.parameters()) + list(plain.parameters()) + list(centered.parameters())
    opt = torch.optim.AdamW(
        [{'params': ad, 'lr': cfg['lr_adapter']}, {'params': hd, 'lr': cfg['lr_head']}],
        weight_decay=cfg['wd'],
    )
    plan = [triplets(fit, rng) for _ in range(cfg['epochs'])]
    total = sum(len(e) for e in plan) // cfg['accum']
    warm = max(1, int(cfg['warmup'] * total))
    sch = torch.optim.lr_scheduler.LambdaLR(
        opt,
        lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))),
    )
    scaler = torch.amp.GradScaler(init_scale=256.0)


### Worker 4 — train loop (optional class-weighted CE)


In [ ]:
%%writefile -a worker.py

    def rep(idx):
        batch = tok.pad({'input_ids': [tokens[i] for i in idx]}, return_tensors='pt').to(dev)
        with torch.autocast('cuda', dtype=torch.float16):
            hs = model(**batch, output_hidden_states=True, use_cache=False).hidden_states
        m = batch['attention_mask'].unsqueeze(-1).float()
        m[:, 0] = 0
        with torch.autocast('cuda', enabled=False):
            return torch.cat(
                [nm((hs[l].float() * m).sum(1) / m.sum(1)) for l, nm in zip(L, norms)],
                -1,
            )

    model.train()
    norms.train()
    step = 0
    skipped = 0
    losses = []
    for ep, batches in enumerate(plan):
        for i, b in enumerate(batches):
            z = rep(b)
            yy = torch.as_tensor(y[b], device=dev)
            loss = nn.functional.cross_entropy(plain(drop(z)), yy, weight=ce_weight) + nn.functional.cross_entropy(
                centered(drop(z - z.mean(0, keepdim=True))), yy, weight=ce_weight
            )
            scaler.scale(loss / cfg['accum']).backward()
            losses.append(float(loss.detach()))
            if (i + 1) % cfg['accum'] == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(ad + hd, 1.0)
                before = scaler.get_scale()
                scaler.step(opt)
                scaler.update()
                if scaler.get_scale() < before:
                    skipped += 1
                opt.zero_grad(set_to_none=True)
                sch.step()
                step += 1
                if step % 30 == 0:
                    print(
                        f'job {job} ep {ep} step {step}/{total} loss {np.mean(losses[-180:]):.4f} '
                        f'skipped {skipped} ({time.time() - t0:.0f}s)',
                        flush=True,
                    )


### Worker 5 — eval + save (`stage1` / `final` write adapter)


In [ ]:
%%writefile -a worker.py

    model.eval()
    norms.eval()
    Z = np.zeros((len(valid), cfg['hidden'] * len(L)), np.float32)
    order = np.argsort([len(tokens[i]) for i in valid], kind='stable')
    with torch.inference_mode():
        for s in range(0, len(order), 6):
            pos = order[s:s + 6]
            Z[pos] = rep(valid[pos]).cpu().numpy()
    assert np.isfinite(Z).all()

    W = {n: t.weight.detach().cpu().numpy() for n, t in (('plain', plain), ('center', centered))}
    B = {n: t.bias.detach().cpu().numpy() for n, t in (('plain', plain), ('center', centered))}
    np.savez_compressed(
        os.path.join(outdir, 'readouts.npz'),
        classes=np.array(LABELS),
        pool_layers=np.array(L),
        plain_coef=W['plain'],
        plain_intercept=B['plain'],
        center_coef=W['center'],
        center_intercept=B['center'],
    )
    np.savez_compressed(
        os.path.join(outdir, 'valid_features.npz'),
        ids=train.id.values[valid],
        features=Z.astype(np.float16),
    )
    lg = Z @ W['plain'].T + B['plain']
    acc = float((lg.argmax(1) == y[valid]).mean())

    # save adapter for stage1, final, and (optionally) fold jobs
    if job in ('stage1', 'final') or cfg.get('save_fold_adapters'):
        model.save_pretrained(os.path.join(outdir, 'adapter'), safe_serialization=True)
        save_file(
            {**{f'norms.{k}': v.detach().cpu().contiguous() for k, v in norms.state_dict().items()}},
            os.path.join(outdir, 'norms.safetensors'),
        )
        tok.save_pretrained(os.path.join(outdir, 'tokenizer'))

    json.dump(
        {
            'job': job,
            'valid_acc': acc,
            'steps': step,
            'skipped': skipped,
            'secs': time.time() - t0,
            'init_adapter': cfg.get('init_adapter'),
            'epochs': cfg['epochs'],
            'train_csv': cfg['train_csv'],
        },
        open(os.path.join(outdir, 'COMPLETE.json'), 'w'),
    )
    print(
        f'job {job} done: valid acc {acc:.4f} steps {step} skipped {skipped} ({time.time() - t0:.0f}s)',
        flush=True,
    )
    del model, base, opt, norms, plain, centered
    torch.cuda.empty_cache()


## Launch

1. Find host `train.csv` + Qwen3.5-9B-Base
2. **Stage 1** on GPU 0 over `pool_train.csv` → `lora_out/job_stage1/` (~2–2.5 h)
3. **Stage 2** on both GPUs (folds + final), warm-started from stage 1 → `lora_out/job_{0..4,final}/`

Submit cells below score `test.csv` from `lora_out/job_final` and write `submission.parquet`.


In [ ]:
T0 = time.time()
DATA_DIR = next(
    p
    for p in [
        pathlib.Path('/kaggle/input/competitions/name-that-llm'),
        pathlib.Path('/kaggle/input/name-that-llm'),
        pathlib.Path('data'),
    ]
    if (p / 'train.csv').exists()
)
host_csv = str(DATA_DIR / 'train.csv')
assert pathlib.Path('pool_train.csv').exists(), 'Run the prepare-pool cell first'
assert pathlib.Path('worker.py').exists(), 'Run the %%writefile worker cells first'

model_dir = None
if MODEL_DIR and pathlib.Path(MODEL_DIR, 'config.json').is_file():
    model_dir = MODEL_DIR

hits = []
for c in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
    try:
        j = json.load(open(c, encoding='utf-8'))
    except Exception:
        continue
    mt = j.get('model_type')
    if mt not in ('qwen3_5_text', 'qwen3_5'):
        continue
    if j.get('hidden_size') != HIDDEN or j.get('num_hidden_layers') != 32:
        continue
    name = c.lower()
    s = 100
    if mt == 'qwen3_5_text':
        s += 30
    if 'base' in name:
        s += 20
    if 'instruct' in name or 'thinking' in name:
        s -= 10
    if '397b' in name or '122b' in name:
        s -= 50
    hits.append((s, os.path.dirname(c)))
hits.sort(key=lambda x: (-x[0], x[1]))
if model_dir is None:
    model_dir = hits[0][1] if hits else None
print('base', model_dir, flush=True)
if len(hits) > 1:
    print('other matches', [h[1] for h in hits[1:]], flush=True)
assert model_dir, (
    'Qwen3.5-9B not found under /kaggle/input. '
    'Attach Qwen3.5-9B-Base (qwen3_5_text, hidden 4096, 32 layers).'
)

OUT_ROOT = 'lora_out'


def run_worker(gpu: int, jobs: list, cfg: dict):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
    log = f'worker_gpu{gpu}_{"_".join(jobs)}.log'
    p = subprocess.Popen(
        [sys.executable, 'worker.py', json.dumps(cfg), ','.join(jobs)],
        env=env,
        stdout=open(log, 'w'),
        stderr=subprocess.STDOUT,
    )
    return p, log


# ----- Stage 1: public pool pretrain (single GPU) -----
cfg1 = dict(
    train_csv='pool_train.csv',
    model_dir=model_dir,
    fold_seed=FOLD_SEED,
    seed=SEED,
    layers=LAYERS,
    keep=KEEP,
    hidden=HIDDEN,
    max_len=MAX_LEN,
    epochs=S1_EPOCHS,
    accum=ACCUM,
    lr_adapter=S1_LR_ADAPTER,
    lr_head=S1_LR_HEAD,
    wd=WD,
    warmup=WARMUP,
    r=R,
    alpha=ALPHA,
    ldrop=LDROP,
    out_root=OUT_ROOT,
    class_weights=S1_USE_CLASS_WEIGHTS,
    init_adapter=None,
)
print('==== STAGE 1: public pool pretrain ====', flush=True)
p1, log1 = run_worker(0, ['stage1'], cfg1)
rc1 = p1.wait()
print(f'---- stage1 (rc {rc1})')
print(open(log1).read()[-5000:])
assert rc1 == 0, 'stage1 failed'
stage1_adapter = str(pathlib.Path(OUT_ROOT) / 'job_stage1' / 'adapter')
assert pathlib.Path(stage1_adapter).is_dir(), stage1_adapter

# ----- Stage 2: host fine-tune from stage1 -----
cfg2 = dict(
    train_csv=host_csv,
    model_dir=model_dir,
    fold_seed=FOLD_SEED,
    seed=SEED,
    layers=LAYERS,
    keep=KEEP,
    hidden=HIDDEN,
    max_len=MAX_LEN,
    epochs=S2_EPOCHS,
    accum=ACCUM,
    lr_adapter=S2_LR_ADAPTER,
    lr_head=S2_LR_HEAD,
    wd=WD,
    warmup=WARMUP,
    r=R,
    alpha=ALPHA,
    ldrop=LDROP,
    out_root=OUT_ROOT,
    class_weights=False,
    init_adapter=stage1_adapter,
)
print('==== STAGE 2: host fine-tune ====', flush=True)
procs = []
for g, jl in S2_JOBS.items():
    procs.append((g, *run_worker(g, jl, cfg2)))
for g, p, log in procs:
    rc = p.wait()
    print(f'---- stage2 worker {g} (rc {rc})')
    print(open(log).read()[-4000:])
    assert rc == 0, f'stage2 gpu {g} failed'

done = sorted(glob.glob(f'{OUT_ROOT}/job_*/COMPLETE.json'))
summary = pd.DataFrame([json.load(open(d)) for d in done])
if not summary.empty:
    cols = [c for c in ['job', 'valid_acc', 'steps', 'skipped', 'secs', 'epochs'] if c in summary.columns]
    summary = summary[cols]
    if 'valid_acc' in summary:
        summary['valid_acc'] = summary['valid_acc'].map(lambda x: f'{x:.4f}')
    if 'secs' in summary:
        summary['secs'] = summary['secs'].map(lambda x: f'{x:.0f}')
print('completed jobs:')
print(summary.to_string(index=False))
print(f'total wall time: {time.time() - T0:.0f}s')
print('next: submit cells below use lora_out/job_final')


## Submit — plain + centred twin + Sinkhorn

Loads `lora_out/job_final` from this run (or an attached training output). Same inference as the 8B submit notebook: merge LoRA into the truncated 9B trunk, mean-pool the trained layers, blend the plain head with the centred twin on TF-IDF topic clusters, then Sinkhorn-balance on the full test set.

Writes `submission.parquet` with `id`, `model`. Interactive 3-row `test.csv` skips Sinkhorn (`len < 50`).


In [ ]:
import gc
import os
from pathlib import Path

import torch
from safetensors.torch import load_file
from torch import nn
from transformers import AutoConfig, AutoModel, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

T_SUB = time.time()
BS = 8
TWIN = dict(thresh=0.85, min_size=2, max_size=18)
W_PLAIN = 1.0
W_TWIN = 0.35


def find_data_csv(name):
    candidates = [
        Path('/kaggle/input/competitions/name-that-llm') / name,
        Path('/kaggle/input/name-that-llm') / name,
        Path('data') / name,
    ]
    for p in candidates:
        if p.exists():
            return p
    for root in ('/kaggle/input', '.'):
        if Path(root).exists():
            hits = sorted(Path(root).rglob(name))
            if hits:
                return hits[0]
    raise FileNotFoundError(name)


def find_job_final():
    patterns = [
        'lora_out/job_final',
        '/kaggle/working/lora_out/job_final',
        '/kaggle/input/**/lora_out/job_final',
        '/kaggle/input/**/job_final',
    ]
    for pat in patterns:
        for hit in sorted(glob.glob(pat, recursive=True)):
            p = Path(hit)
            if (p / 'adapter').is_dir() and (p / 'readouts.npz').exists() and (p / 'norms.safetensors').exists():
                return p
    raise FileNotFoundError(
        'Need job_final/{adapter/, readouts.npz, norms.safetensors}. '
        'Train first, or attach a finished 9B LoRA24 output.'
    )


def find_qwen35_9b_base():
    if 'model_dir' in globals() and model_dir:
        return model_dir
    hits = []
    for c in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
        try:
            j = json.load(open(c, encoding='utf-8'))
        except Exception:
            continue
        mt = j.get('model_type')
        if mt not in ('qwen3_5_text', 'qwen3_5'):
            continue
        if j.get('hidden_size') != HIDDEN or j.get('num_hidden_layers') != 32:
            continue
        name = c.lower()
        s = 100
        if mt == 'qwen3_5_text':
            s += 30
        if 'base' in name:
            s += 20
        if 'instruct' in name or 'thinking' in name:
            s -= 10
        hits.append((s, os.path.dirname(c)))
    hits.sort(key=lambda x: (-x[0], x[1]))
    if not hits:
        raise FileNotFoundError('Attach Qwen3.5-9B-Base (qwen3_5_text, hidden 4096, 32 layers)')
    return hits[0][1]


def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


CKPT = find_job_final()
BASE_DIR = find_qwen35_9b_base()
test = pd.read_csv(find_data_csv('test.csv'))
test['text'] = test['text'].fillna('').astype(str)

with np.load(CKPT / 'readouts.npz', allow_pickle=False) as z:
    LABELS = [str(x) for x in z['classes'].tolist()]
    POOL_LAYERS = [int(x) for x in z['pool_layers'].tolist()]
    PLAIN_W = z['plain_coef'].astype(np.float64)
    PLAIN_B = z['plain_intercept'].astype(np.float64)
    CENTER_W = z['center_coef'].astype(np.float64)
    CENTER_B = z['center_intercept'].astype(np.float64)

assert POOL_LAYERS == LAYERS, (POOL_LAYERS, LAYERS)
K = len(LABELS)
assert PLAIN_W.shape == (K, HIDDEN * len(LAYERS)), PLAIN_W.shape
print('checkpoint', CKPT)
print('base', BASE_DIR)
print('labels', LABELS)
print('test', test.shape, flush=True)


### Load trunk, merge LoRA, pool features

Do **not** replace `model.norm` with Identity — this recipe kept the final RMSNorm, same as the 8B host run.


In [ ]:
def load_trunk_and_adapter(base_dir, adapter_dir, keep):
    mc = AutoConfig.from_pretrained(base_dir, local_files_only=True)
    mc.num_hidden_layers, mc.use_cache = keep, False
    if getattr(mc, 'layer_types', None) is not None:
        mc.layer_types = mc.layer_types[:keep]
    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    base = AutoModel.from_pretrained(
        base_dir,
        config=mc,
        quantization_config=quant,
        device_map={'': 0},
        local_files_only=True,
        low_cpu_mem_usage=True,
        torch_dtype=torch.float16,
    )
    return PeftModel.from_pretrained(base, str(adapter_dir)).eval()


@torch.no_grad()
def extract_features(texts, bs=BS):
    tok_dir = CKPT / 'tokenizer'
    tok = AutoTokenizer.from_pretrained(str(tok_dir if tok_dir.is_dir() else BASE_DIR), local_files_only=True)
    tok.padding_side = 'right'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = load_trunk_and_adapter(BASE_DIR, CKPT / 'adapter', KEEP)

    norms = nn.ModuleList([nn.LayerNorm(HIDDEN) for _ in LAYERS]).to('cuda:0').float()
    nsd = load_file(str(CKPT / 'norms.safetensors'))
    norms.load_state_dict({k.split('norms.', 1)[-1]: v for k, v in nsd.items()})

    ids = tok(
        list(texts),
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LEN,
    )['input_ids']
    ids = [x + [tok.eos_token_id] for x in ids]
    order = np.argsort([len(x) for x in ids], kind='stable')
    Z = np.zeros((len(ids), HIDDEN * len(LAYERS)), np.float32)

    for s in range(0, len(order), bs):
        idx = order[s : s + bs]
        enc = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to('cuda:0')
        hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
        m = enc['attention_mask'].unsqueeze(-1).float()
        m[:, 0] = 0
        den = m.sum(1).clamp(min=1)
        Z[idx] = torch.cat(
            [nm((hs[l].float() * m).sum(1) / den) for l, nm in zip(LAYERS, norms)],
            -1,
        ).cpu().numpy()

    del model, norms
    torch.cuda.empty_cache()
    gc.collect()
    return Z


Z = extract_features(test.text.tolist(), bs=BS)
print('Z', Z.shape, f'({time.time() - T_SUB:.0f}s)', flush=True)
assert np.isfinite(Z).all()


### Centred twin (topic clusters)

Cluster test passages by TF-IDF. Inside clusters of size 2–18, subtract the cluster mean and score with the **center** head.


In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import TfidfVectorizer

Xt = TfidfVectorizer(stop_words='english', sublinear_tf=True).fit_transform(test.text.tolist())
if len(test) >= 2:
    D = np.clip(1 - (Xt @ Xt.T).toarray(), 0, None)
    np.fill_diagonal(D, 0)
    groups = fcluster(
        linkage(squareform(D, checks=False), method='average'),
        TWIN['thresh'],
        criterion='distance',
    )
    del D
else:
    groups = np.ones(len(test), dtype=int)

size = np.bincount(groups)[groups]
twin_rows = (size >= TWIN['min_size']) & (size <= TWIN['max_size'])
print(
    f'topic clusters: {int((size == 1).sum())} singletons, '
    f'{int(twin_rows.sum())} passages in clusters of 2-18',
    flush=True,
)

plain_logits = Z @ PLAIN_W.T + PLAIN_B
LOGIT = W_PLAIN * np.log(np.clip(softmax(plain_logits), 1e-9, 1))

if twin_rows.any():
    Zc = Z.copy()
    for g in np.unique(groups[twin_rows]):
        Zc[groups == g] -= Zc[groups == g].mean(0)
    center_logits = Zc[twin_rows] @ CENTER_W.T + CENTER_B
    LOGIT[twin_rows] += W_TWIN * np.log(np.clip(softmax(center_logits), 1e-9, 1))

print(f'heads applied ({time.time() - T_SUB:.0f}s)', flush=True)


### Sinkhorn + `submission.parquet`

Rescale so each of the 18 classes gets ~equal mass on the full test set. Skipped for the 3-row interactive `test.csv`.


In [ ]:
P = softmax(LOGIT)
if len(test) >= 50:
    target = np.full(K, len(P) / K)
    for _ in range(100):
        P *= target / np.clip(P.sum(0), 1e-12, None)
        P /= np.clip(P.sum(1, keepdims=True), 1e-12, None)

sub = pd.DataFrame(
    {
        'id': test['id'].astype(str),
        'model': np.array(LABELS)[P.argmax(1)],
    }
)
assert sub['id'].is_unique
assert set(sub['model']).issubset(set(LABELS))
sub.to_parquet('submission.parquet', index=False)

print(sub['model'].value_counts().sort_index().to_string())
print(f'done in {time.time() - T_SUB:.0f}s -> submission.parquet ({len(sub)} rows)')
